# Лаб 2. Random Forest

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.datasets import fetch_openml

from source.model import RandomForestClassifier

## Данные

In [2]:
data = fetch_openml('heart-statlog', version=1, as_frame=True)
df = data.frame

target_col = df.columns[-1]
feature_names = [c for c in df.columns if c != target_col]

X = df[feature_names].values.astype(float)
y_raw = df[target_col].values
classes_map = {v: i for i, v in enumerate(np.unique(y_raw))}
y = np.array([classes_map[v] for v in y_raw])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'train={len(y_train)} test={len(y_test)} features={X.shape[1]}')

train=216 test=54 features=13


## Grid Search по OOB

In [3]:
grc = GridSearchCV(RandomForestClassifier(), {
    'n_estimators': [10, 50, 150],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt']
}, cv=3, verbose=1)

grc.fit(X_train, y_train)
print(f'лучшие параметры: {grc.best_params_}')
print(f'лучший score: {grc.best_score_:.4f}')

Fitting 3 folds for each of 81 candidates, totalling 243 fits
лучшие параметры: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 10}
лучший score: 0.7450


## Обучение с лучшими параметрами

In [4]:
t0 = time.time()
my_rf = RandomForestClassifier(**grc.best_params_)
my_rf.fit(X_train, y_train)
my_time = time.time() - t0

my_pred = my_rf.predict(X_test)
my_acc = accuracy_score(y_test, my_pred)
print(f'accuracy={my_acc:.4f} oob={my_rf.oob_score_:.4f} time={my_time:.3f}s')
print(confusion_matrix(y_test, my_pred))

accuracy=0.8148 oob=0.7321 time=0.007s
[[24  6]
 [ 4 20]]


## Важность признаков (OOB^j)

In [5]:
importances = my_rf._calculate_feature_importance()

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(imp_df.to_string(index=False))

                             feature  importance
                 fasting_blood_sugar    0.077634
resting_electrocardiographic_results    0.075948
              resting_blood_pressure    0.072575
                   serum_cholestoral    0.069202
                                 age    0.065829
             exercise_induced_angina    0.045592
                               chest    0.043905
         maximum_heart_rate_achieved    0.042219
                                 sex    0.028727
                               slope    0.023668
                             oldpeak    0.016922
             number_of_major_vessels    0.001744
                                thal   -0.042103


## Сравнение со sklearn

In [18]:
t0 = time.time()
sk_rf = SklearnRF(oob_score=True, **grc.best_params_)
sk_rf.fit(X_train, y_train)
sk_time = time.time() - t0

sk_pred = sk_rf.predict(X_test)
sk_acc = accuracy_score(y_test, sk_pred)
print(f'sklearn: accuracy={sk_acc:.4f} oob={sk_rf.oob_score_:.4f} time={sk_time:.3f}s')
print(confusion_matrix(y_test, sk_pred))

sklearn: accuracy=0.8519 oob=0.8009 time=0.015s
[[25  5]
 [ 3 21]]


/Users/uvuv/ML_Algorithms_2/.venv/lib/python3.14/site-packages/sklearn/ensemble/_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


## Итого

In [21]:
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': sk_rf.feature_importances_
}).sort_values('importance', ascending=False)

imp_df

,feature,importance
11,number_of_major_vessels,0.161573
2,chest,0.161110
9,oldpeak,0.132499
10,slope,0.099920
12,thal,0.088634
1,sex,0.080049
0,age,0.068251
7,maximum_heart_rate_achieved,0.064204
8,exercise_induced_angina,0.063071
3,resting_blood_pressure,0.045367


In [19]:
pd.DataFrame({
    'модель': ['Custom RF', 'sklearn RF'],
    'accuracy': [my_acc, sk_acc],
    'OOB score': [my_rf.oob_score_, sk_rf.oob_score_],
    'время (с)': [my_time, sk_time],
})

,модель,accuracy,OOB score,время (с)
0,Custom RF,0.814815,0.732056,0.007176
1,sklearn RF,0.851852,0.800926,0.015060
